# 🖐️ Hand Gesture Classification — Notebook 2: EDA & Insights

## 📘 Tổng quan Dự án
### 🎯 Bài toán
Xây dựng mô hình phân loại cử chỉ tay dựa trên tọa độ của 21 điểm Landmark (x, y) được trích xuất từ MediaPipe.
*   **Input:** Tọa độ 21 điểm landmark (42 đặc trưng: p0_x, p0_y, ..., p20_x, p20_y).
*   **Output:** Nhãn cử chỉ (1-one, 2-two, 3-three, 4-fist, 5-palm, 7-victory).

### 📑 Mục lục
1. [⚙️ Thiết lập Môi trường](#1.-Thiết-lập-Môi-trường)
2. [📊 Tổng quan Dữ liệu](#2.-Tổng-quan-Dữ-liệu)
3. [📈 Phân tích Phân bố Nhãn (Target Distribution)](#3.-Phân-tích-Phân-bố-Nhãn)
4. [📍 Phân tích Đặc trưng Landmark (Feature Analysis)](#4.-Phân-tích-Đặc-trưng-Landmark)
5. [🔗 Phân tích Tương quan (Correlation)](#5.-Phân-tích-Tương-quan)
6. [🌌 Trực quan hóa Không gian t-SNE](#6.-Trực-quan-hóa-t-SNE)
7. [✍️ Phân tích Chữ ký đặc trưng (Mean Profile)](#7.-Phân-tích-Chữ-ký-đặc-trưng)
8. [📝 Tổng kết EDA](#8.-Tổng-kết-EDA)

---

# <a id="1.-Thiết-lập-Môi-trường"></a>1. ⚙️ Thiết lập Môi trường

In [1]:
import os
import sys
sys.path.append("../")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from motion_detection.utils.get_missing import *

sns.set(style="whitegrid")

DATA_PATH = "motion_detection/models/hand_gestures.csv"
IMAGE_DIR = "Images"

os.makedirs(IMAGE_DIR, exist_ok=True)

---

# <a id="2.-Tổng-quan-Dữ-liệu"></a>2. 📊 Tổng quan Dữ liệu

### 2.1. Tải dữ liệu

*Lưu ý: Trong quá trình thu thập dữ liệu, thư mục không mong muốn là `archive`. Ta tiến hành loại bỏ các nhãn nhiễu này để đảm bảo độ chính xác cho phân tích dữ liệu.*

In [2]:
df = pd.read_csv(DATA_PATH)

print("Kích thước dữ liệu:", df.shape)
df.head()

Kích thước dữ liệu: (14467, 43)


,p0_x,p0_y,p1_x,p1_y,p2_x,p2_y,p3_x,p3_y,p4_x,p4_y,...,p16_y,p17_x,p17_y,p18_x,p18_y,p19_x,p19_y,p20_x,p20_y,label
0,0.0,0.0,0.371057,-0.195043,0.500531,-0.377706,0.606193,-0.502968,0.677487,-0.612329,...,-0.374041,0.081629,-0.352000,0.484942,-0.397403,0.499546,-0.328662,0.393402,-0.297586,1-one
1,0.0,0.0,0.111712,0.001854,0.217892,-0.152969,0.230506,-0.314415,0.146643,-0.360131,...,-0.112527,-0.051856,-0.445408,-0.031372,-0.467644,-0.011077,-0.316911,0.007867,-0.199143,1-one
2,0.0,0.0,-0.194745,-0.077442,-0.356908,-0.205940,-0.401998,-0.335197,-0.312624,-0.437184,...,-0.200738,0.282745,-0.407561,0.143727,-0.390567,0.069926,-0.274221,0.053462,-0.202830,1-one
3,0.0,0.0,-0.110246,-0.116826,-0.187975,-0.270936,-0.177418,-0.421474,-0.118392,-0.537246,...,-0.321678,0.113902,-0.422740,0.101905,-0.504417,0.076403,-0.380569,0.069558,-0.328978,1-one
4,0.0,0.0,0.201101,-0.115647,0.296182,-0.290313,0.266590,-0.424632,0.157655,-0.488081,...,-0.275580,-0.186903,-0.393544,-0.160508,-0.458688,-0.103403,-0.360976,-0.075393,-0.286336,1-one


### 2.2. Kiểm tra thông tin chung

In [3]:
display("🔎 Thông tin dữ liệu:")
df.info()

'🔎 Thông tin dữ liệu:'

<class 'pandas.DataFrame'>
RangeIndex: 14467 entries, 0 to 14466
Data columns (total 43 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   p0_x    14467 non-null  float64
 1   p0_y    14467 non-null  float64
 2   p1_x    14467 non-null  float64
 3   p1_y    14467 non-null  float64
 4   p2_x    14467 non-null  float64
 5   p2_y    14467 non-null  float64
 6   p3_x    14467 non-null  float64
 7   p3_y    14467 non-null  float64
 8   p4_x    14467 non-null  float64
 9   p4_y    14467 non-null  float64
 10  p5_x    14467 non-null  float64
 11  p5_y    14467 non-null  float64
 12  p6_x    14467 non-null  float64
 13  p6_y    14467 non-null  float64
 14  p7_x    14467 non-null  float64
 15  p7_y    14467 non-null  float64
 16  p8_x    14467 non-null  float64
 17  p8_y    14467 non-null  float64
 18  p9_x    14467 non-null  float64
 19  p9_y    14467 non-null  float64
 20  p10_x   14467 non-null  float64
 21  p10_y   14467 non-null  float64
 22  p11_x   1

### 2.3. Kiểm tra missing values và trùng lặp

In [4]:
print(f"Số dòng bị trùng lặp: {df.duplicated().sum()}")
print(f"Số giá trị thiếu: {df.isnull().sum().sum()}")
get_missings_percentage(df, df.columns)

Số dòng bị trùng lặp: 0
Số giá trị thiếu: 0
Không có dữ liệu bị thiếu trong các cột được chọn.


### 2.4. Thống kê mô tả

In [5]:
display(df.describe())

,p0_x,p0_y,p1_x,p1_y,p2_x,p2_y,p3_x,p3_y,p4_x,p4_y,...,p16_x,p16_y,p17_x,p17_y,p18_x,p18_y,p19_x,p19_y,p20_x,p20_y
count,14467.0,14467.0,14467.000000,14467.000000,14467.000000,14467.000000,14467.000000,14467.000000,14467.000000,14467.000000,...,14467.000000,14467.000000,14467.000000,14467.000000,14467.000000,14467.000000,14467.000000,14467.000000,14467.000000,14467.000000
mean,0.0,0.0,0.010826,-0.108998,0.019760,-0.254831,0.020262,-0.363354,0.017761,-0.416015,...,0.008030,-0.398757,0.010058,-0.335945,0.002624,-0.403390,-0.000869,-0.342574,0.000596,-0.296330
std,0.0,0.0,0.221814,0.088160,0.357774,0.190561,0.395866,0.284570,0.427922,0.347143,...,0.211891,0.433254,0.250009,0.264736,0.284925,0.346714,0.263644,0.350928,0.252427,0.361227
min,0.0,0.0,-0.638899,-0.453046,-1.000000,-0.702573,-1.000000,-0.921905,-1.000000,-1.000000,...,-1.000000,-1.000000,-0.772092,-1.000000,-0.949088,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
25%,0.0,0.0,-0.171486,-0.166343,-0.235485,-0.359260,-0.166290,-0.481100,-0.183738,-0.545845,...,-0.093107,-0.573994,-0.205176,-0.416085,-0.226282,-0.526811,-0.184388,-0.493087,-0.149880,-0.437100
50%,0.0,0.0,0.058824,-0.116274,0.050338,-0.291294,0.018982,-0.425013,0.011625,-0.484946,...,0.005304,-0.394623,-0.029465,-0.372421,-0.026416,-0.452389,-0.019994,-0.382966,-0.009667,-0.317600
75%,0.0,0.0,0.187750,-0.077455,0.262702,-0.224040,0.215010,-0.337976,0.217624,-0.398937,...,0.112973,-0.302989,0.230431,-0.333656,0.232661,-0.395250,0.178874,-0.317627,0.147571,-0.250726
max,0.0,0.0,0.709833,0.382519,1.000000,0.674281,1.000000,0.872815,1.000000,1.000000,...,1.000000,1.000000,0.890001,0.896677,1.000000,1.000000,0.999784,0.984240,1.000000,1.000000


---

# <a id="3.-Phân-tích-Phân-bố-Nhãn"></a>3. 📈 Phân tích Phân bố Nhãn (Target Distribution)

In [6]:
gesture_counts = df['label'].value_counts().sort_index()

fig = px.bar(
    x=gesture_counts.index,
    y=gesture_counts.values,
    labels={'x': 'Gesture', 'y': 'Số lượng ảnh'},
    title='Phân bố số lượng gesture có trong Dataset',
    color=gesture_counts.values,
    color_continuous_scale='Viridis'
)

fig.update_layout(
    template='plotly_white',
    height=500,
    width=800
)

fig.update_traces(
    texttemplate='%{y:,}',
    textposition='outside'
)

fig.show()

print(f"\nTổng số ảnh: {df.shape[0]}")
print(f"Tổng số gesture: {df['label'].nunique()}")

# LƯU ẢNH
fig.write_image(f"{IMAGE_DIR}/01_gesture_distribution.png")


Tổng số ảnh: 14467
Tổng số gesture: 6


📝 **Nhận xét**

**Gesture Distribution (Phân bố số lượng cử chỉ)**

*   **Nhóm chiếm ưu thế:** Các cử chỉ `2-two`, `3-three`, và `7-victory` có số lượng mẫu lớn nhất, đều vượt mức 3,100 mẫu.
*   **Nhóm thiểu số:** Các cử chỉ `1-one`, `4-open_close`, và `5-rotate` có số lượng ít hơn đáng kể, chỉ dao động trong khoảng 1,500–1,600 mẫu.
*   **Tỉ lệ chênh lệch:** Số lượng mẫu giữa nhóm cao nhất và thấp nhất chênh lệch khoảng 2 lần.
*   **→ Mất cân bằng dữ liệu (Class Imbalance):** Dữ liệu không đồng đều giữa các lớp, cần sử dụng kỹ thuật chia tập dữ liệu **Stratified Sampling** và ưu tiên đánh giá bằng chỉ số **F1-Score** thay vì chỉ dùng Accuracy.

---

# <a id="4.-Phân-tích-Đặc-trưng-Landmark"></a>4. 📍 Phân tích Đặc trưng Landmark (Feature Analysis)

In [7]:
fig = px.box(
    df, 
    x="label", 
    y="p8_y", 
    color="label",
    title="Biểu đồ 2: So sánh phân bố tọa độ P8_Y giữa các loại cử chỉ",
    points="outliers"
)
fig.show()

# LƯU ẢNH
fig.write_image(f"{IMAGE_DIR}/02_feature_boxplot_p8y.png")

📝 **Nhận xét**

**Feature Distribution (Phân bố đặc trưng P8_Y - Tọa độ Y đỉnh ngón trỏ)**

*   **Sự khác biệt rõ rệt:** Cử chỉ `4-open_close` có dải phân bố (hộp) nằm cao hơn hẳn so với các nhóm còn lại. Điều này cho thấy tọa độ Y của ngón trỏ là đặc trưng quan trọng để nhận diện cử chỉ nắm tay.
*   **Sự chồng lấn (Overlap):** Các cử chỉ `1, 2, 3, 5, 7` có phần thân hộp nằm sát nhau ở vùng đáy (giá trị gần -1.0). Điều này có nghĩa là nếu chỉ dựa vào một tọa độ P8_Y, máy sẽ rất khó phân biệt được các cử chỉ này.
*   **Giá trị ngoại lệ (Outliers):** Xuất hiện rất nhiều điểm dữ liệu rời rạc trải dài từ -0.5 đến 1.0 ở tất cả các lớp. 
*   **→ Vấn đề dữ liệu:** Các outlier này có thể do tay bị xoay nhiều hướng hoặc lỗi cảm biến khi thu thập. 
*   **→ Hành động:** Cần thực hiện **Scaling/Normalization** hoặc sử dụng các đặc trưng tương đối (như khoảng cách giữa các khớp) thay vì tọa độ tuyệt đối để giảm bớt sự ảnh hưởng của việc tay nằm xa/gần hoặc lệch trong khung hình.

---

# <a id="5.-Phân-tích-Tương-quan"></a>5. 🔗 Phân tích Tương quan (Correlation)


In [8]:
corr = df.drop(columns=['label']).corr()

fig = px.imshow(
    corr,
    color_continuous_scale='RdBu',
    title="Landmark Feature Correlation Heatmap"
)

fig.update_layout(
    template='plotly_white',
    height=800,
    width=800
)

fig.show()
fig.write_image(f"{IMAGE_DIR}/03_correlation_heatmap.png")

📝 **Nhận xét**

**Landmark Correlation (Tương quan giữa các đặc trưng)**

*   **Tương quan dương cực mạnh (Dark Blue):** Các khối màu xanh đậm xuất hiện dày đặc, cho thấy sự tương quan rất cao giữa các điểm landmark kế cận nhau (ví dụ: các khớp trên cùng một ngón tay). Khi một khớp di chuyển, các khớp còn lại có xu hướng di chuyển cùng hướng.
*   **Cấu trúc bàn cờ (Checkerboard pattern):** Biểu đồ thể hiện rõ sự tách biệt giữa tọa độ X và Y. Các tọa độ X thường tương quan mạnh với nhau và tương tự với tọa độ Y, nhưng tương quan giữa X và Y của cùng một điểm thường thấp hơn.
*   **Sự dư thừa dữ liệu (Redundancy):** Do các điểm landmark có độ tương quan quá cao, tập dữ liệu chứa nhiều thông tin lặp lại.

**→ Giảm chiều dữ liệu:** Đây là dấu hiệu cho thấy việc áp dụng các kỹ thuật như **PCA** hoặc **t-SNE** sẽ cực kỳ hiệu quả. Nó giúp loại bỏ các đặc trưng dư thừa, giảm nhiễu và tăng tốc độ huấn luyện mô hình mà không làm mất đi nhiều thông tin cốt lõi.

**→ Hành động:** Có thể cân nhắc chỉ giữ lại các điểm đầu ngón tay (tips) hoặc sử dụng kỹ thuật trích xuất đặc trưng để giảm số lượng biến đầu vào từ 42 xuống mức thấp hơn.

---

# <a id="6.-Trực-quan-hóa-t-SNE"></a>6. 🌌 Trực quan hóa Không gian t-SNE

### 6.1 Tiền xử lý

In [9]:
X = df.drop(columns=['label'])
y = df['label']

# 3. Chuẩn hóa dữ liệu (Z-score Scaling)
# Điều này giúp các tọa độ x, y có tầm ảnh hưởng ngang nhau
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Dữ liệu đã sẵn sàng: {X_scaled.shape}")

Dữ liệu đã sẵn sàng: (14467, 42)


### 6.2 Trực quan hoá

In [10]:
tsne = TSNE(
    n_components=2, 
    perplexity=30, 
    max_iter=1000, 
    random_state=42,
    init='pca', 
    learning_rate='auto'
)

X_tsne = tsne.fit_transform(X_scaled)

# Tạo DataFrame để vẽ biểu đồ
tsne_df = pd.DataFrame({
    "Dim_1": X_tsne[:, 0],
    "Dim_2": X_tsne[:, 1],
    "Gesture": y.astype(str) # Ép kiểu string để Plotly hiểu là dữ liệu phân loại
})

# Vẽ biểu đồ Scatter tương tác
fig = px.scatter(
    tsne_df, 
    x="Dim_1", 
    y="Dim_2", 
    color="Gesture",
    title="Biểu đồ t-SNE: Phân cụm toàn bộ các loại cử chỉ tay",
    labels={"Dim_1": "Đặc trưng t-SNE 1", "Dim_2": "Đặc trưng t-SNE 2"},
    hover_name="Gesture",
    opacity=0.7,
    color_discrete_sequence=px.colors.qualitative.Antique
)

fig.update_layout(
    template='plotly_white',
    legend_title="Loại Cử Chỉ",
    width=900,
    height=700
)

fig.show()
fig.write_image(f"{IMAGE_DIR}/04_tsne_projection.png")

📝 **Nhận xét**

**t-SNE Projection (Trực quan hóa toàn bộ không gian dữ liệu)**

*   **Khả năng phân cụm:** Dữ liệu có khả năng phân tách khá tốt. Các cử chỉ tạo thành những "đảo" dữ liệu riêng biệt, cho thấy các đặc trưng landmark đủ mạnh để phân loại.
*   **Nhóm tách biệt rõ rệt (Dễ nhận diện):** Cử chỉ `4-open_close` (màu xanh xám) nằm tách hẳn về phía bên trái. Cử chỉ `5-rotate` (màu xanh rêu) cũng tạo thành các cụm khá độc lập ở phía trên và dưới.
    *   → Các lớp này sẽ có độ chính xác (Precision/Recall) rất cao khi huấn luyện mô hình.
*   **Nhóm chồng lấn (Dễ nhầm lẫn):** Có sự trộn lẫn đáng kể giữa các lớp `2-two`, `3-three` và `7-victory` ở khu vực trung tâm và phía bên phải biểu đồ. 
    *   → Điều này hợp lý về mặt hình học vì các cử chỉ này đều có các ngón tay xòe ra tương tự nhau (2 ngón vs 3 ngón). Đây sẽ là những lớp mà mô hình dễ dự đoán sai nhất.
*   **Sự phân tán:** Một số màu (như nhãn 3) xuất hiện ở nhiều cụm nhỏ khác nhau thay vì một cụm duy nhất.
    *   → Cho thấy sự đa dạng trong tập dữ liệu (ví dụ: cùng một cử chỉ nhưng có nhiều góc xoay tay hoặc kích thước tay khác nhau).
    
**→ Hành động:** Đối với các nhóm hay bị chồng lấn (`2, 3, 7`), cần xem xét trích xuất thêm đặc trưng về **góc giữa các ngón tay** hoặc **khoảng cách tương đối** để giúp mô hình phân biệt tốt hơn thay vì chỉ dùng tọa độ Landmark thô.

---

# <a id="7.-Phân-tích-Chữ-ký-đặc-trưng"></a>7. ✍️ Phân tích Chữ ký đặc trưng (Mean Profile)

In [11]:
# Tính trung bình tọa độ cho mỗi nhãn
df_grouped = df.groupby('label').mean()

# Chọn các điểm đại diện cho 5 đầu ngón tay (y-axis)
fingertip_coords = ['p4_y', 'p8_y', 'p12_y', 'p16_y', 'p20_y']
df_fingers = df_grouped[fingertip_coords].T

# Tên ngón tay cho đẹp
finger_names = ["Thumb", "Index", "Middle", "Ring", "Pinky"]

fig = go.Figure()

for column in df_fingers.columns:
    fig.add_trace(
        go.Scatter(
            x=finger_names,
            y=df_fingers[column],
            mode="lines+markers",
            name=f"Gesture {column}"
        )
    )

fig.update_layout(
    title="Biểu đồ 5: Hình dáng đặc trưng (Profile) của các cử chỉ",
    xaxis_title="Đầu ngón tay (Cái → Út)",
    yaxis_title="Giá trị trung bình Tọa độ Y",
    template="plotly_white",
    width=900,
    height=600
)

fig.show()
fig.write_image(f"{IMAGE_DIR}/05_mean_profile.png")

📝 **Nhận xét**

**Mean Feature Profile (Chữ ký hình học đặc trưng của cử chỉ)**

*   **Dấu vân tay dữ liệu (Geometric Signature):** Mỗi đường biểu diễn một "hình dáng trung bình" của bàn tay cho từng loại cử chỉ. Sự khác biệt về hình dạng các đường phản ánh cách các ngón tay co/duỗi khác nhau giữa các lớp.
*   **Sự tương đồng gây nhiễu:** 
    *   Đường biểu diễn của cử chỉ `2-two` (đỏ) và `7-victory` (xanh dương nhạt) gần như **trùng khít nhau** ở tất cả các ngón.
    *   → **Insight:** Điều này giải thích tại sao biểu đồ t-SNE trước đó lại có sự chồng lấn mạnh giữa hai lớp này. Về mặt tọa độ trung bình, chúng có cấu trúc hình học cực kỳ giống nhau.
*   **Các lớp đặc trưng (Unique Profiles):**
    *   `4-open_close` (tím): Có đường biểu diễn "phẳng" nhất và nằm ở mức cao (vùng ngón tay co lại).
    *   `5-rotate` (cam): Có xu hướng nằm thấp hẳn xuống ở các ngón giữa và ngón nhẫn (vùng ngón tay xòe rộng).
*   **Các điểm nút quan trọng (Key Discriminators):** Ngón **Ring (Nhẫn)** và **Pinky (Út)** là hai vị trí có sự phân tách lớn nhất giữa các đường biểu diễn.

**→ Hành động:** 
 *   Thay vì sử dụng tọa độ thô, nên tính toán **độ dốc (slope)** hoặc **độ chênh lệch tọa độ** giữa các ngón tay kề nhau. 
*   Thông tin về ngón út và ngón nhẫn sẽ là "chìa khóa" để mô hình phân biệt các cử chỉ có cấu trúc gần giống nhau như 1, 2, 3 và 7.

---

# <a id="8.-Tổng-kết-EDA"></a>8. 📝 Tổng kết EDA

1.  **Chất lượng dữ liệu:** Không có giá trị thiếu, dữ liệu đã được làm sạch cơ bản.
2.  **Đặc điểm nhãn:** Có sự mất cân bằng nhẹ giữa nhóm cử chỉ số (1, 2, 3) và cử chỉ trạng thái bàn tay.
3.  **Thách thức:** Sự tương đồng hình học giữa `2-two` và `7-victory` là khó khăn lớn nhất cho mô hình.
4.  **Hướng đi tiếp theo:** 
    *   Sử dụng **StandardScaler** để chuẩn hóa dữ liệu.
    *   Áp dụng **Feature Engineering** (tính góc ngón tay) để cải thiện độ phân tách.